In [1]:
import numpy as np
import scanpy
from Ensembl_converter import EnsemblConverter

In [2]:
ann = scanpy.read_h5ad('b84def55-a776-4aa4-a9a6-7aab8b973086.h5ad')

In [3]:
ann

AnnData object with n_obs × n_vars = 483152 × 58482
    obs: 'tissue_in_publication', 'assay_ontology_term_id', 'donor_id', 'anatomical_information', 'n_counts_UMIs', 'n_genes', 'cell_ontology_class', 'free_annotation', 'manually_annotated', 'compartment', 'sex_ontology_term_id', 'disease_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_type', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length'
    uns: '_scvi', '_training_mode', 'citation', 'dendrogram_cell_type_tissue', 'dendrogram_computational_compartment_assignment', 'dendrogra

In [4]:
cell_types = ann.obs.cell_type.to_numpy()
unique_cell_types = list(set(cell_types))

In [5]:
ensembl_ids = ann.var.index.to_numpy()
ensembl_ids

array(['ENSG00000223972', 'ENSG00000227232', 'ENSG00000278267', ...,
       'ENSG00000198727', 'ENSG00000210195', 'ENSG00000210196'],
      dtype=object)

In [6]:
means, stds = [], []
for cell_type in unique_cell_types:
    mask = (cell_types == cell_type)
    mean = np.squeeze(np.asarray(ann.X[mask, :].mean(axis=0)))
    means.append(mean)
    ann.X[mask, :].todense()
    std = np.mean(np.square(ann.X[mask, :].todense() - mean[np.newaxis, :]), axis=0)
    stds.append(std)
means = np.asarray(means)
stds = np.asarray(stds)

100%|████████████████████████████████████████████████████████████████████████████████| 161/161 [06:28<00:00,  2.41s/it]


In [7]:
means.shape

(161, 58482)

In [7]:
mapping = {}
with open('ensembl-ids.txt', 'r') as f:
    for line in f:
        elements = line.rstrip().split()
        if len(elements) == 1:
            mapping[elements[0]] = ''
        elif len(elements) == 2:
            mapping[elements[0]] = elements[1]

In [8]:
gene_symbols = np.asarray([mapping[ensembl_id] for ensembl_id in ensembl_ids], dtype=object)

In [9]:
np.savez(
    'expression-profiles.npz',
    mu=means,
    sigma=stds,
    ensembl_ids=ensembl_ids,
    cell_types=unique_cell_types,
    gene_symbols=gene_symbols
)